# 03 · Construcción de la segmentación RFM y del indicador de churn

Calcula Recency, Frequency y Monetary sobre los 903 clientes con historial de compra, asigna los segmentos de valor, define el indicador de riesgo de churn y exporta `clientes_rfm.csv` para su carga en MySQL (informe técnico, apartado 4.4).

## 1. Carga de la capa clean

In [1]:
import pandas as pd
import numpy as np
import os

ruta = os.path.join('..', '..', 'data', 'clean')

clientes = pd.read_csv(os.path.join(ruta, 'clientes_clean.csv'))
transacciones = pd.read_csv(os.path.join(ruta, 'transacciones_clean.csv'), parse_dates=['fecha'])

print(len(clientes), len(transacciones))
print(transacciones['importe'].sum().round(2))
print(transacciones['fecha'].min(), transacciones['fecha'].max())

1000 8000
1640308.39
2023-01-01 00:00:00 2025-12-31 00:00:00


## 2. Fecha de referencia y cálculo de R, F y M

La fecha de referencia se fija de forma explícita en el 1 de enero de 2026, el día siguiente a la última transacción registrada, en lugar de derivarse en cada ejecución.

In [2]:
fecha_ref = pd.Timestamp('2026-01-01')

rfm = (transacciones.groupby('customer_id')
       .agg(ultima_compra=('fecha', 'max'),
            frecuencia=('transaction_id', 'count'),
            monetario=('importe', 'sum'))
       .reset_index())

rfm['recencia'] = (fecha_ref - rfm['ultima_compra']).dt.days

print(len(rfm))
print(rfm[['recencia', 'frecuencia', 'monetario']].describe().round(2))
print(rfm['monetario'].sum().round(2))

903
       recencia  frecuencia  monetario
count    903.00      903.00     903.00
mean     211.54        8.86    1816.51
std      242.41        8.20    1722.46
min        1.00        1.00    -241.18
25%       42.00        3.00     570.71
50%      118.00        7.00    1309.27
75%      285.00       12.00    2482.81
max     1096.00       51.00   10426.34
1640308.39


In [3]:
rfm.head()

,customer_id,ultima_compra,frecuencia,monetario,recencia
0,C0001,2025-05-04,3,633.07,242
1,C0002,2025-10-11,7,1279.16,82
2,C0003,2025-03-06,5,1140.96,301
3,C0004,2025-01-12,7,1223.93,354
4,C0005,2025-11-24,13,2873.84,38


## 3. Scores de 1 a 5

En Frequency los quintiles se calculan sobre el rango (`rank(method='first')`) porque la frecuencia tiene pocos valores distintos y los empates impiden grupos equilibrados. El desempate es determinista: la tabla agregada está ordenada por `customer_id`.

In [4]:
# Recency: menos días = mejor → escala invertida
rfm['R_score'] = pd.qcut(rfm['recencia'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency: sobre el rango, por los empates
rfm['F_score'] = pd.qcut(rfm['frecuencia'].rank(method='first'), 5,
                         labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary
rfm['M_score'] = pd.qcut(rfm['monetario'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

for col in ['R_score', 'F_score', 'M_score']:
    print(col, rfm[col].value_counts().sort_index().tolist())

R_score [181, 180, 181, 176, 185]
F_score [181, 180, 181, 180, 181]
M_score [181, 180, 181, 180, 181]


## 4. Segmentación

- **Alto Valor:** F + M ≥ 9 **y** R ≥ 4.
- **Bajo Valor:** F + M ≤ 5 **o** R ≤ 2.
- **Medio Valor:** el resto.

In [5]:
rfm['valor_fm'] = rfm['F_score'] + rfm['M_score']

condiciones = [
    (rfm['valor_fm'] >= 9) & (rfm['R_score'] >= 4),   # Alto
    (rfm['valor_fm'] <= 5) | (rfm['R_score'] <= 2),   # Bajo
]
rfm['segmento_rfm'] = np.select(condiciones, ['Alto Valor', 'Bajo Valor'],
                                default='Medio Valor')

print(rfm['segmento_rfm'].value_counts())
print(rfm.groupby('segmento_rfm')[['recencia', 'frecuencia', 'monetario']].mean().round(2))

segmento_rfm
Bajo Valor     487
Medio Valor    267
Alto Valor     149
Name: count, dtype: int64
              recencia  frecuencia  monetario
segmento_rfm                                 
Alto Valor       30.62       22.38    4591.61
Bajo Valor      342.24        3.92     796.08
Medio Valor      74.12       10.32    2129.09


## 5. Riesgo de churn

Un cliente está **en riesgo de churn** cuando su recencia al 1 de enero de 2026 supera el **percentil 70** de la distribución de recencia de los 903 compradores. El umbral se calcula en el código, no se escribe a mano: con esta fecha de referencia y esta ventana (2023-2025) resulta en **234 días**, lo que equivale a una última compra anterior al 12 de mayo de 2025.

Se usa un percentil y no un plazo fijo porque la ventana de observación es acotada (la recencia no puede superar los 1.096 días): el corte se ajusta a la muestra de clientes efectivamente observada. Si cambia la fecha de referencia o la ventana, el umbral se recalcula solo.

`churn_riesgo` indica **riesgo**, no abandono consumado. Su capacidad predictiva se valida fuera de muestra en el apartado 5.5 del informe.

In [6]:
# Umbral = percentil 70 de la recencia de los compradores.
# La recencia es entera, así que tomar la parte entera del percentil no cambia la clasificación.
umbral_churn = int(rfm['recencia'].quantile(0.70))
rfm['churn_riesgo'] = (rfm['recencia'] > umbral_churn).astype(int)

print('Umbral (días):', umbral_churn)
print('En riesgo si la última compra es anterior a:', (fecha_ref - pd.Timedelta(days=umbral_churn)).date())
print(rfm['churn_riesgo'].sum(), rfm['churn_riesgo'].mean().round(4))
print(rfm.groupby('segmento_rfm')['churn_riesgo'].sum())

Umbral (días): 234
En riesgo si la última compra es anterior a: 2025-05-12
269 0.2979
segmento_rfm
Alto Valor       0
Bajo Valor     269
Medio Valor      0
Name: churn_riesgo, dtype: int64


In [7]:
# Controles: el umbral no depende del método de interpolación del percentil
for m in ['linear', 'lower', 'higher', 'nearest']:
    print(m, rfm['recencia'].quantile(0.70, interpolation=m))
print('Compradores con recencia <= umbral:', (rfm['recencia'] <= umbral_churn).mean().round(4))

linear 234.0
lower 234
higher 234
nearest 234
Compradores con recencia <= umbral: 0.7021


## 6. Incorporación de los 97 clientes sin transacciones

Reciben recencia 999 (valor placeholder), frecuencia y monetario 0, segmento `Sin Transacciones` y `churn_riesgo = 0`: son leads no convertidos, no clientes fugados.

In [8]:
clientes_rfm = clientes[['customer_id']].merge(
    rfm[['customer_id', 'recencia', 'frecuencia', 'monetario',
         'segmento_rfm', 'churn_riesgo']],
    on='customer_id', how='left')

sin_compra = clientes_rfm['recencia'].isna()
clientes_rfm.loc[sin_compra, ['recencia', 'frecuencia', 'monetario', 'churn_riesgo']] = [999, 0, 0, 0]
clientes_rfm.loc[sin_compra, 'segmento_rfm'] = 'Sin Transacciones'

clientes_rfm = clientes_rfm.astype({'recencia': int, 'frecuencia': int, 'churn_riesgo': int})
clientes_rfm['monetario'] = clientes_rfm['monetario'].round(2)
clientes_rfm['fecha_calculo'] = fecha_ref.date()

print(len(clientes_rfm))
print(clientes_rfm['segmento_rfm'].value_counts())

1000
segmento_rfm
Bajo Valor           487
Medio Valor          267
Alto Valor           149
Sin Transacciones     97
Name: count, dtype: int64


## 7. Preparación y exportación para MySQL

In [9]:
clientes_rfm = clientes_rfm.astype({
    'customer_id':   'string',
    'recencia':      'int64',
    'frecuencia':    'int64',
    'monetario':     'float64',
    'segmento_rfm':  'string',
    'churn_riesgo':  'int8',
})
clientes_rfm['monetario'] = clientes_rfm['monetario'].round(2)
clientes_rfm['fecha_calculo'] = pd.to_datetime(clientes_rfm['fecha_calculo'])

# Controles antes de exportar
print(clientes_rfm.dtypes)
print('Nulos:', clientes_rfm.isna().sum().sum())
print('Largo máx. customer_id:', clientes_rfm['customer_id'].str.len().max())
print('Largo máx. segmento_rfm:', clientes_rfm['segmento_rfm'].str.len().max())
print('Monetario fuera de DECIMAL(10,2):', (clientes_rfm['monetario'].abs() >= 1e8).sum())

customer_id             string
recencia                 int64
frecuencia               int64
monetario              float64
segmento_rfm            string
churn_riesgo              int8
fecha_calculo    datetime64[s]
dtype: object
Nulos: 0
Largo máx. customer_id: 5
Largo máx. segmento_rfm: 17
Monetario fuera de DECIMAL(10,2): 0


In [10]:
clientes_rfm.to_csv(
    os.path.join(ruta, 'clientes_rfm.csv'),
    index=False,
    encoding='utf-8',
    float_format='%.2f',
    date_format='%Y-%m-%d',
)

# Verificación
check = pd.read_csv(os.path.join(ruta, 'clientes_rfm.csv'))
print(check.shape)
print(check['segmento_rfm'].value_counts())

(1000, 7)
segmento_rfm
Bajo Valor           487
Medio Valor          267
Alto Valor           149
Sin Transacciones     97
Name: count, dtype: int64
